# Lab 1.2 · Three time scales

Day 1 · Session 2 (Seeing structure) · 25 min.

You look at Al-Rimal's daily demand at three time scales (nine years, one year, four weeks) before any
model. Each scale shows a different pattern. You shade Ramadan and the two Eids, measure the weekday
profile and the Eid dip of every year, and find one dip that no calendar explains.

**AI policy: AI-tutor.** An assistant may explain a plot; you write the measurements.

**Reference state.** `data/regions.csv` and `data/calendar.csv`. Nothing from another lab.

**Tasks**
- [ ] 1 · Plot nine years and name the trend and the annual shape (5')
- [ ] 2 · Plot one year against temperature (5')
- [ ] 3 · Plot four weeks and compute the weekday profile (6')
- [ ] 4 · Shade Ramadan and Eid and measure the Eid dip per year (6')
- [ ] 5 · Find the dip that is not a holiday (3')
- [ ] ✱ Stretch · Repeat for Al-Jabal, a cool mountain region, and compare

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-304.git"
    if not Path("/content/forecasting").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/forecasting"], check=True)
    os.chdir("/content/forecasting/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("shams.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import shams as U

plt = U.style()

df = U.region("R01")          # Al-Rimal: demand (MWh per day), temp_c, temp_normal
y = df["demand"]
cal = U.calendar()
print(df.tail(3))

## 1 · Nine years (5')

At this scale you see the trend and the annual shape, and nothing about weekdays.

In [ ]:
fig, ax = plt.subplots()
ax.plot(y.index, y, lw=0.6, color="#16324f")
ax.plot(y.rolling(365, center=True).mean(), color="#e4572e", lw=2, label="365-day mean")
ax.set_title("Al-Rimal, daily demand, 2017 to 2025", loc="left"); ax.set_ylabel("MWh per day"); ax.legend(frameon=False)
plt.show()

✏️ **Exercise** · Return the average growth per year, in percent, from the mean of 2017 to the mean of 2025
(eight years of compounding).

In [ ]:
def annual_growth(y: pd.Series) -> float:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
# Check, with a safety net: the cells below still run when the exercise is not done yet
try:
    growth = annual_growth(y)
except NotImplementedError:
    growth = U.reference("d1_2")["growth"]
U.check(2.0 < growth < 3.5, f"demand grows about {growth:.1f}% a year")
peak_month = int(y.groupby(y.index.month).mean().idxmax())
ratio = float(y.loc["2025-07"].mean() / y.loc["2025-01"].mean())
print(f"highest month: {peak_month}; July 2025 is {ratio:.1f} times January 2025")

## 2 · One year, against temperature (5')

In [ ]:
one = df.loc["2024"]
fig, ax = plt.subplots()
ax.plot(one.index, one.demand, color="#16324f", lw=1)
ax2 = ax.twinx(); ax2.plot(one.index, one.temp_c, color="#e4572e", lw=0.8, alpha=0.7); ax2.grid(False)
ax.set_ylabel("MWh per day"); ax2.set_ylabel("°C"); ax.set_title("Al-Rimal, 2024: demand (dark) and temperature", loc="left")
plt.show()

✏️ **Exercise** · Demand follows cooling, and cooling starts above about 22 °C. Return the correlation between
demand and "cooling degrees" (`temp_c` minus 22, never below zero) over the whole series.

In [ ]:
def cooling_correlation(df: pd.DataFrame) -> float:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    corr = cooling_correlation(df)
except NotImplementedError:
    corr = U.reference("d1_2")["corr"]
U.check(corr > 0.9, f"correlation of demand with cooling degrees: {corr:.2f}")

## 3 · Four weeks, and the weekday profile (6')

✏️ **Exercise** · Return the mean demand of each weekday divided by the overall mean, as a Series indexed
`Mon` to `Sun`. Use the last three years only, so growth does not blur it.

In [ ]:
def weekday_profile(y: pd.Series) -> pd.Series:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    prof = weekday_profile(y)
except NotImplementedError:
    prof = pd.Series(U.reference("d1_2")["profile"])
print(prof.round(3).to_string())
U.check(prof.idxmin() == "Fri" and prof["Sat"] < prof["Sun"], "Friday is the lowest day, then Saturday: the weekend is Friday and Saturday")

four = y.loc["2025-10-05":"2025-11-01"]
fig, ax = plt.subplots()
ax.plot(four.index, four, marker="o", color="#16324f")
for d in four.index[four.index.dayofweek == 4]:
    ax.axvline(d, color="#e4572e", alpha=0.4)
ax.set_title("Four weeks in autumn 2025; the lines are Fridays", loc="left"); plt.show()

## 4 · Ramadan, Eid, and the dip that moves (6')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6.5))
for ax, yr in zip(axes, (2019, 2022, 2025)):
    s = y.loc[f"{yr}-02-01":f"{yr}-09-15"]
    ax.plot(s.index, s, color="#16324f", lw=1)
    U.shade_events(ax, s.index[0], s.index[-1])
    ax.set_title(f"{yr}: Ramadan (light), Eid al-Fitr and Eid al-Adha (dark)", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

✏️ **Exercise** · For one year, find the first day of Eid al-Fitr (`eid_fitr == 1` and `eid_weight == 1`) and
return its date and the dip: demand on that day against the mean of the same weekday one week before and
one week after, in percent (negative for a dip). Taking both neighbours cancels the spring warming.

In [ ]:
def eid_dip(y: pd.Series, cal: pd.DataFrame, year: int) -> tuple:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    dips = {yr: eid_dip(y, cal, yr) for yr in range(2017, 2026)}
    dips = {yr: (d.strftime("%Y-%m-%d"), round(float(v), 1)) for yr, (d, v) in dips.items()}
except NotImplementedError:
    dips = {int(k): tuple(v) for k, v in U.reference("d1_2")["dips"].items()}
table = pd.DataFrame(dips, index=["first day of Eid al-Fitr", "dip %"]).T
print(table.to_string())
U.check(table["dip %"].astype(float).max() < -5, f"every Eid al-Fitr is a dip: from {table['dip %'].astype(float).max()}% to {table['dip %'].astype(float).min()}%")
shift = pd.to_datetime(table.iloc[:, 0]).dt.dayofyear.diff().dropna()
print("days earlier each year:", (-shift).astype(int).tolist())

**Look at.** The dip is as regular as a season, but it arrives 10 to 12 days earlier each Gregorian year. A
model with a 365-day season will look for it on the wrong days.

## 5 · The dip that is not a holiday (3')

In Al-Rimal a cool week can cut demand by a quarter, so a drop alone proves nothing. Look for a drop the
weather does not explain.

✏️ **Exercise** · Compute each day's change against the mean of the same weekday one week before and one
week after, for demand and for temperature. Among the days with no calendar event (`event == ""`) that were
not cooler than their two neighbours, return the date of the largest drop in demand.

In [ ]:
def strange_day(r: pd.DataFrame, cal: pd.DataFrame) -> pd.Timestamp:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    odd = strange_day(df, cal)
except NotImplementedError:
    odd = pd.Timestamp(U.reference("d1_2")["odd"])
odd_drop = round(float((y[odd] / ((y[odd - pd.Timedelta(days=7)] + y[odd + pd.Timedelta(days=7)]) / 2) - 1) * 100), 1)
print("largest drop the weather does not explain:", odd.date(), f"{odd_drop}%")
U.check(odd == pd.Timestamp("2023-03-14"), "the drop of 14 March 2023 is in no calendar: ask operations (a sandstorm outage)")
U.keep("d1_2", dict(growth=growth, corr=corr, profile=prof.round(4).to_dict(), dips=dips, odd=odd, odd_drop=odd_drop,
                    peak_month=peak_month, ratio=ratio))

## Checkpoint

<details><summary>1 · Which pattern can you see only in the four-week plot?</summary>
The weekly one: Friday low, Saturday a little low. At nine years it is invisible.</details>

<details><summary>2 · Why is "the same date last year" a poor guide to the Eid dip?</summary>
The Hijri year is about 11 days shorter than the Gregorian year, so Eid falls on a different date each year.</details>

<details><summary>3 · What do you do with 14 March 2023?</summary>
Ask before deleting. Operations confirm an outage: demand was not low, supply was. Keep a note of it; a model
should not learn it as a pattern.</details>

## Carry forward

Three facts later labs use: the weekend is Friday and Saturday; demand follows cooling degrees above 22 °C;
Eid moves. Lab 1.3 measures how strong each pattern is.